# SAR-to-EO Image Translation — GalaxEye Assignment
**Model:** Pix2Pix U-Net + PatchGAN  
**Loss:** L1 + Adversarial + FFT Frequency + VGG Perceptual  
**Dataset:** Sentinel-1&2 terrain-split (Kaggle)


In [ ]:
# CELL 1: Check GPU
import subprocess, os, torch
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 2: Install packages (dependency conflict warnings are NORMAL — not errors)
!pip install lpips pytorch-fid scikit-image -q

In [ ]:
# CELL 3: Clone OR update repo (safe to run multiple times)
import os, subprocess, shutil

REPO_DIR = '/kaggle/working/sar2eo'
REPO_URL = 'https://github.com/Trafalgar-2006/sar2eo.git'

if os.path.exists(REPO_DIR):
    # Already cloned — just pull latest
    print('Repo exists — pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'master'], check=True)
else:
    print('Cloning repo...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
os.makedirs('outputs', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('logs', exist_ok=True)
print('Working directory:', os.getcwd())
print('Files:', sorted(os.listdir('.')))

In [ ]:
# CELL 4: Auto-detect dataset path
import os
from pathlib import Path

TERRAIN_NAMES = ['barren', 'grassland', 'agricultural', 'urban']
DATASET_PATH = None

print('Scanning /kaggle/input/...')
for item in sorted(Path('/kaggle/input').iterdir()):
    if not item.is_dir():
        continue
    try:
        subdirs = [d.name.lower() for d in item.iterdir() if d.is_dir()]
    except PermissionError:
        continue
    hits = [t for t in TERRAIN_NAMES if t in subdirs]
    print(f'  {item.name}/ -> {subdirs[:6]}')
    if len(hits) >= 2:
        DATASET_PATH = str(item)
        print(f'  *** FOUND TERRAIN DATASET HERE: {DATASET_PATH} ***')

if DATASET_PATH is None:
    print('\nDataset NOT found — please add the Sentinel-1&2 dataset and restart.')
else:
    print(f'\nDATASET_PATH = {DATASET_PATH}')

In [ ]:
# CELL 5: Update config
import yaml

# If Cell 4 did not find the dataset, manually set the path here:
# DATASET_PATH = '/kaggle/input/datasets'

if DATASET_PATH is None:
    raise RuntimeError('DATASET_PATH is None. Set it manually above or attach the dataset and restart.')

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['dataset_type']   = 'kaggle'
cfg['data']['kaggle_root']    = DATASET_PATH
cfg['data']['subset_size']    = None
cfg['training']['batch_size'] = 4
cfg['training']['epochs']     = 100
cfg['training']['num_workers'] = 2

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Config saved:')
print(f"  kaggle_root : {cfg['data']['kaggle_root']}")
print(f"  batch_size  : {cfg['training']['batch_size']}")
print(f"  epochs      : {cfg['training']['epochs']}")

In [ ]:
# CELL 6: Smoke test — 2 epochs, 100 pairs. MUST PASS before training.
import yaml, subprocess, copy

with open('config.yaml') as f:
    cfg_orig = yaml.safe_load(f)

cfg_smoke = copy.deepcopy(cfg_orig)
cfg_smoke['data']['subset_size']   = 100
cfg_smoke['training']['epochs']    = 2
cfg_smoke['training']['val_freq']  = 1
cfg_smoke['training']['save_freq'] = 2
cfg_smoke['active_ablation']       = 'full'

with open('config_smoke.yaml', 'w') as f:
    yaml.dump(cfg_smoke, f)

print('Running smoke test (2 epochs, 100 pairs)...')
result = subprocess.run(
    ['python', 'train.py', '--config', 'config_smoke.yaml', '--ablation', 'full'],
    capture_output=False
)
if result.returncode == 0:
    print('\nSmoke test PASSED -- proceeding to full training')
else:
    raise RuntimeError('Smoke test FAILED — fix the error above before running full training')

In [ ]:
# CELL 7a: Config A — L1 only (baseline)
import subprocess
print('Training Config A: L1 only...')
result = subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_only'])
print('Config A done!' if result.returncode == 0 else 'Config A FAILED')

In [ ]:
# CELL 7b: Config B — L1 + Adversarial
import subprocess
print('Training Config B: L1 + Adversarial...')
result = subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_adv'])
print('Config B done!' if result.returncode == 0 else 'Config B FAILED')

In [ ]:
# CELL 7c: Config C — L1 + Adversarial + FFT
import subprocess
print('Training Config C: L1 + Adversarial + FFT...')
result = subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_adv_fft'])
print('Config C done!' if result.returncode == 0 else 'Config C FAILED')

In [ ]:
# CELL 7d: Config D — Full model (MAIN)
import subprocess
print('Training Config D (MAIN): Full loss stack...')
result = subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'full'])
print('Config D done!' if result.returncode == 0 else 'Config D FAILED')

In [ ]:
# CELL 8: Evaluate all configs on test split
import os, subprocess
for ablation in ['l1_only', 'l1_adv', 'l1_adv_fft', 'full']:
    ckpt = f'checkpoints/{ablation}/best.pth'
    if not os.path.exists(ckpt):
        print(f'Skipping {ablation} — no checkpoint found')
        continue
    print(f'\n=== Evaluating {ablation} ===')
    subprocess.run(['python', 'eval.py', '--config', 'config.yaml', '--weights', ckpt, '--split', 'test'])
print('\nAll done!')

In [ ]:
# CELL 9: Zip outputs for download
import shutil, os
shutil.make_archive('/kaggle/working/sar2eo_results', 'zip', '.', 'outputs')
shutil.make_archive('/kaggle/working/sar2eo_checkpoints', 'zip', '.', 'checkpoints')
for f in ['/kaggle/working/sar2eo_results.zip', '/kaggle/working/sar2eo_checkpoints.zip']:
    mb = os.path.getsize(f) / 1e6 if os.path.exists(f) else 0
    print(f'  {f}  ({mb:.1f} MB)')
print('Download from the Output panel on the right.')